In [1]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q datasets
print("✅ ready")

Mounted at /content/drive
✅ ready


In [2]:
from datasets import load_dataset
import pandas as pd

print("Downloading drug reviews (3-5 min)...")
ds = load_dataset('lewtun/drug-reviews')
df = pd.concat([
    ds['train'].to_pandas(),
    ds['test'].to_pandas()
], ignore_index=True)
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

top_drugs = df['drugName'].str.lower().value_counts().head(50).index.tolist()

sampled = []
for drug in top_drugs:
    drug_df = df[df['drugName'].str.lower() == drug].copy()
    high = drug_df[drug_df['rating'] >= 7].sample(
        min(250, len(drug_df[drug_df['rating'] >= 7])), random_state=42)
    low  = drug_df[drug_df['rating'] <= 3].sample(
        min(250, len(drug_df[drug_df['rating'] <= 3])), random_state=42)
    sampled.append(pd.concat([high, low]))

sample_df = pd.concat(sampled, ignore_index=True).sample(
    frac=1, random_state=42).reset_index(drop=True)
sample_df['drug_name'] = sample_df['drugName'].str.lower()

print(f"✅ Sampled: {len(sample_df)} rows, {sample_df['drug_name'].nunique()} drugs")
sample_df.to_csv('/content/drive/MyDrive/DrugRadar/data/raw/drug_reviews_25k.csv', index=False)
print("✅ Saved drug_reviews_25k.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.jsonl:   0%|          | 0.00/97.7M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/32.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/161297 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/53766 [00:00<?, ? examples/s]

Total rows: 215063
Columns: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']
✅ Sampled: 21861 rows, 50 drugs
✅ Saved drug_reviews_25k.csv


In [3]:
import torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset

MODEL_PATH   = "/content/drive/MyDrive/DrugRadar/models/checkpoints/best_model"
REVIEWS_PATH = "/content/drive/MyDrive/DrugRadar/data/raw/drug_reviews_25k.csv"
OUTPUT_PATH  = "/content/drive/MyDrive/DrugRadar/data/predictions_25k.csv"
BATCH_SIZE   = 32

print("Loading model from Drive...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"✅ Model loaded on {device}")

class ReviewDataset(Dataset):
    def __init__(self, texts): self.texts = texts
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(str(self.texts[i])[:512], truncation=True,
                        max_length=128, padding='max_length', return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

df     = pd.read_csv(REVIEWS_PATH)
loader = DataLoader(ReviewDataset(df['review'].tolist()), batch_size=BATCH_SIZE)
all_preds, all_confs = [], []

print(f"Running inference on {len(df)} reviews...")
with torch.no_grad():
    for i, batch in enumerate(loader):
        batch   = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs   = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_preds.extend(probs[:, 1] > 0.5)
        all_confs.extend(probs[:, 1])
        if i % 50 == 0:
            print(f"  {i}/{len(loader)} batches done ({i/len(loader)*100:.0f}%)")

df['pred_label']     = [int(p) for p in all_preds]
df['confidence']     = [round(float(c), 4) for c in all_confs]
df['norm_rating']    = (df['rating'] - 1) / 9
df['severity_score'] = df.apply(
    lambda r: round(r['confidence'] * (1 - r['norm_rating']), 4)
    if r['pred_label'] == 1 else 0.0, axis=1)
df['severity_bucket'] = df['severity_score'].apply(
    lambda s: 'critical' if s >= 0.6 else 'moderate' if s >= 0.3 else 'weak')

print(f"\n✅ Done!")
print(f"ADE rate : {df['pred_label'].mean()*100:.1f}%")
print(f"Critical : {(df['severity_bucket']=='critical').sum()}")
print(f"Moderate : {(df['severity_bucket']=='moderate').sum()}")

df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Saved predictions_25k.csv to Drive!")

Loading model from Drive...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded on cuda
Running inference on 21861 reviews...
  0/684 batches done (0%)
  50/684 batches done (7%)
  100/684 batches done (15%)
  150/684 batches done (22%)
  200/684 batches done (29%)
  250/684 batches done (37%)
  300/684 batches done (44%)
  350/684 batches done (51%)
  400/684 batches done (58%)
  450/684 batches done (66%)
  500/684 batches done (73%)
  550/684 batches done (80%)
  600/684 batches done (88%)
  650/684 batches done (95%)

✅ Done!
ADE rate : 18.2%
Critical : 2002
Moderate : 272
✅ Saved predictions_25k.csv to Drive!
